# (08) Jobs: BALLS (T=16)

project = ```iP-VAE```, host = ```chewie```, device = ```any```

**Motivation**: <br>

Create jobs for all the BALLS dataset fits.

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_IterativeVAE'))
from figures.analysis import plot_convergence
from figures.imgs import plot_weights
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = 'Dropbox/git/_IterativeVAE/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

[
    'copyfits.sh',
    'fit_model.sh',
    'kill_screens.sh',
    'resume_fit.sh',
    'run_sessions.sh',
    'test_tqdm.py',
    'test_tqdm.sh'
]

## chewie (```T=16```)

```<grad|lin>```

In [4]:
host = 'chewie'
_cleanup(save_dir, host)

scripts_mach = collections.defaultdict(list)
tot = 0

In [5]:
# n_seeds = 5
# seeds = range(1, n_seeds + 1)

model_type = 'poisson'
seq_len = 16

npix_list = [16, 64]

latent_dim_mult = [
    0.5, 1, 1.5, 2,
    3, 4, 8, 16,
]
latent_dims = {
    n: [int(m * n) for m in latent_dim_mult]
    for n in npix_list
}

betas_mult = [
    0.5, 0.75, 1, 1.25,
    1.5, 2, 3, 4,
]
betas = [
    m * seq_len for
    m in betas_mult
]

print(f"latent_dims: {latent_dims}\nbetas: {betas}")

latent_dims: {16: [8, 16, 24, 32, 48, 64, 128, 256], 64: [32, 64, 96, 128, 192, 256, 512, 1024]}
betas: [8.0, 12.0, 16, 20.0, 24.0, 32, 48, 64]

In [6]:
for beta in betas:
    for npix, latent_dims_list in latent_dims.items():
        for n_latents in latent_dims_list:
            arg = ' '.join([
                f"--seq_len {seq_len}",
                f"--n_latents {n_latents}",
                f"--kl_beta {beta}",
                f"--comment t-{seq_len}_b-{beta:0.3g}_k-{n_latents}",
            ])
            gpu_i = tot % 2
            kws = dict(
                device=gpu_i,
                dataset=f"BALLS{npix}",
                archi='grad|lin',
                model=model_type,
                args=arg,
                seed=1,
            )
            scripts_mach[gpu_i].append(job_runner_script(**kws))
            tot += 1

In [7]:
print(tot)

128

In [8]:
scripts_mach = dict(scripts_mach)
print({k: len(v) for k, v in scripts_mach.items()})

{0: 64, 1: 64}

### Save

In [9]:
n_fits = 4

for gpu_i, scripts in scripts_mach.items():
    scripts_divided = divide_list(scripts, n_fits)
    for fit_i, strings_list in enumerate(scripts_divided):
        # sort so BALLS64 don't conicide together
        sorted_strings = sorted(
            strings_list,
            key=lambda x: "BALLS64" in x,
            reverse=fit_i % 2 == 0,
        )
        combined = ' && '.join(sorted_strings)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'chewie-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 8.0 --comment 
t-16_b-8_k-32 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 96 --kl_beta 8.0 --comment 
t-16_b-8_k-96 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 192 --kl_beta 8.0 --comment 
t-16_b-8_k-192 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 512 --kl_beta 8.0 --comment 
t-16_b-8_k-512 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 12.0 --comment 
t-16_b-12_k-32 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 96 --kl_beta 12.0 --comment 
t-16_b-12_k-96 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 192 --kl_beta 12.0 --comment 
t-16_b-12_k-192 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 512 --kl_beta 12.0 --comment 
t-16_b-12_k-512 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 8 --kl_beta 8.0 --comment 
t-16_b-8_k-8 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 24 --kl_beta 8.0 --comment 
t-16_b-8_k-24 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 48 --kl_beta 8.0 --comment 
t-16_b-8_k-48 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 8.0 --comment 
t-16_b-8_k-128 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 8 --kl_beta 12.0 --comment 
t-16_b-12_k-8 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 24 --kl_beta 12.0 --comment 
t-16_b-12_k-24 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 48 --kl_beta 12.0 --comment 
t-16_b-12_k-48 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 12.0 --comment 
t-16_b-12_k-128

[PROGRESS] 'chewie-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 8 --kl_beta 16 --comment 
t-16_b-16_k-8 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 24 --kl_beta 16 --comment 
t-16_b-16_k-24 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 48 --kl_beta 16 --comment 
t-16_b-16_k-48 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 16 --comment 
t-16_b-16_k-128 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 8 --kl_beta 20.0 --comment 
t-16_b-20_k-8 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 24 --kl_beta 20.0 --comment 
t-16_b-20_k-24 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 48 --kl_beta 20.0 --comment 
t-16_b-20_k-48 && 
./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 20.0 --comment 
t-16_b-20_k-128 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 16 --comment 
t-16_b-16_k-32 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 96 --kl_beta 16 --comment 
t-16_b-16_k-96 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 192 --kl_beta 16 --comment 
t-16_b-16_k-192 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 512 --kl_beta 16 --comment 
t-16_b-16_k-512 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 20.0 --comment 
t-16_b-20_k-32 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 96 --kl_beta 20.0 --comment 
t-16_b-20_k-96 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 192 --kl_beta 20.0 --comment 
t-16_b-20_k-192 && 
./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 512 --kl_beta 20.0 --comment 
t-16_b-20_k-512

[PROGRESS] 'chewie-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

...

./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 16 --comment 
t-16_b-16_k-16 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 16 --comment 
t-16_b-16_k-32 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 16 --comment 
t-16_b-16_k-64 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 16 --comment 
t-16_b-16_k-256 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 20.0 --comment 
t-16_b-20_k-16 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 20.0 --comment 
t-16_b-20_k-32 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 20.0 --comment 
t-16_b-20_k-64 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 20.0 --comment 
t-16_b-20_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 16 --comment 
t-16_b-16_k-64 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 16 --comment 
t-16_b-16_k-128 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 16 --comment 
t-16_b-16_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 16 --comment 
t-16_b-16_k-1024 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 20.0 --comment 
t-16_b-20_k-64 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 20.0 --comment 
t-16_b-20_k-128 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 20.0 --comment 
t-16_b-20_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 20.0 --comment 
t-16_b-20_k-1024

[PROGRESS] 'chewie-cuda1-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 24.0 --comment 
t-16_b-24_k-64 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 24.0 --comment 
t-16_b-24_k-128 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 24.0 --comment 
t-16_b-24_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 24.0 --comment 
t-16_b-24_k-1024 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 32 --comment 
t-16_b-32_k-64 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 32 --comment 
t-16_b-32_k-128 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 32 --comment 
t-16_b-32_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 32 --comment 
t-16_b-32_k-1024 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 24.0 --comment 
t-16_b-24_k-16 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 24.0 --comment 
t-16_b-24_k-32 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 24.0 --comment 
t-16_b-24_k-64 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 24.0 --comment 
t-16_b-24_k-256 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 32 --comment 
t-16_b-32_k-16 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 32 --comment 
t-16_b-32_k-32 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 32 --comment 
t-16_b-32_k-64 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 32 --comment 
t-16_b-32_k-256

[PROGRESS] 'chewie-cuda1-fit3.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 48 --comment 
t-16_b-48_k-16 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 48 --comment 
t-16_b-48_k-32 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 48 --comment 
t-16_b-48_k-64 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 48 --comment 
t-16_b-48_k-256 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 64 --comment 
t-16_b-64_k-16 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 64 --comment 
t-16_b-64_k-32 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 64 --comment 
t-16_b-64_k-64 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 64 --comment 
t-16_b-64_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 48 --comment 
t-16_b-48_k-64 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 48 --comment 
t-16_b-48_k-128 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 48 --comment 
t-16_b-48_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 48 --comment 
t-16_b-48_k-1024 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 64 --comment 
t-16_b-64_k-64 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 64 --comment 
t-16_b-64_k-128 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 64 --comment 
t-16_b-64_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 64 --comment 
t-16_b-64_k-1024

In [10]:
sorted_strings

["./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 48 --comment t-16_b-48_k-16",
 "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 48 --comment t-16_b-48_k-32",
 "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 48 --comment t-16_b-48_k-64",
 "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 48 --comment t-16_b-48_k-256",
 "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 64 --comment t-16_b-64_k-16",
 "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 64 --comment t-16_b-64_k-32",
 "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 64 --comment t-16_b-64_k-64",
 "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_

Print one to check

In [11]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 48 --comment 
t-16_b-48_k-16 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 48 --comment 
t-16_b-48_k-32 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 48 --comment 
t-16_b-48_k-64 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 48 --comment 
t-16_b-48_k-256 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 64 --comment 
t-16_b-64_k-16 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 64 --comment 
t-16_b-64_k-32 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 64 --comment 
t-16_b-64_k-64 && 
./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 64 --comment 
t-16_b-64_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 48 --comment 
t-16_b-48_k-64 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 48 --comment 
t-16_b-48_k-128 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 48 --comment 
t-16_b-48_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 48 --comment 
t-16_b-48_k-1024 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 64 --comment 
t-16_b-64_k-64 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 64 --comment 
t-16_b-64_k-128 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 64 --comment 
t-16_b-64_k-256 && 
./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 64 --comment 
t-16_b-64_k-1024

In [12]:
print(scripts_mach)

{
    0: [
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 8 --kl_beta 8.0 
--comment t-16_b-8_k-8",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 24 --kl_beta 8.0 
--comment t-16_b-8_k-24",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 48 --kl_beta 8.0 
--comment t-16_b-8_k-48",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 8.0 
--comment t-16_b-8_k-128",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 8.0 
--comment t-16_b-8_k-32",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 96 --kl_beta 8.0 
--comment t-16_b-8_k-96",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 192 --kl_beta 8.0 
--comment t-16_b-8_k-192",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 512 --kl_beta 8.0 
--comment t-16_b-8_k-512",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 8 --kl_beta 12.0 
--comment t-16_b-12_k-8",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 24 --kl_beta 12.0 
--comment t-16_b-12_k-24",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 48 --kl_beta 12.0 
--comment t-16_b-12_k-48",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 12.0 
--comment t-16_b-12_k-128",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 12.0 
--comment t-16_b-12_k-32",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 96 --kl_beta 12.0 
--comment t-16_b-12_k-96",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 192 --kl_beta 12.0 
--comment t-16_b-12_k-192",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 512 --kl_beta 12.0 
--comment t-16_b-12_k-512",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 8 --kl_beta 16 
--comment t-16_b-16_k-8",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 24 --kl_beta 16 
--comment t-16_b-16_k-24",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 48 --kl_beta 16 
--comment t-16_b-16_k-48",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 16 
--comment t-16_b-16_k-128",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 16 
--comment t-16_b-16_k-32",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 96 --kl_beta 16 
--comment t-16_b-16_k-96",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 192 --kl_beta 16 
--comment t-16_b-16_k-192",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 512 --kl_beta 16 
--comment t-16_b-16_k-512",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 8 --kl_beta 20.0 
--comment t-16_b-20_k-8",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 24 --kl_beta 20.0 
--comment t-16_b-20_k-24",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 48 --kl_beta 20.0 
--comment t-16_b-20_k-48",
        "./fit_model.sh '0' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 20.0 
--comment t-16_b-20_k-128",
        "./fit_model.sh '0' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 20.0 
--com

In [13]:
print(scripts_divided)

[
    [
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 8.0 
--comment t-16_b-8_k-16",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 8.0 
--comment t-16_b-8_k-32",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 8.0 
--comment t-16_b-8_k-64",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 8.0 
--comment t-16_b-8_k-256",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 8.0 
--comment t-16_b-8_k-64",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 8.0 
--comment t-16_b-8_k-128",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 8.0 
--comment t-16_b-8_k-256",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 8.0 
--comment t-16_b-8_k-1024",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 12.0 
--comment t-16_b-12_k-16",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 12.0 
--comment t-16_b-12_k-32",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 12.0 
--comment t-16_b-12_k-64",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 12.0 
--comment t-16_b-12_k-256",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 12.0 
--comment t-16_b-12_k-64",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 12.0 
--comment t-16_b-12_k-128",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 12.0 
--comment t-16_b-12_k-256",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 12.0 
--comment t-16_b-12_k-1024"
    ],
    [
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 16 
--comment t-16_b-16_k-16",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 16 
--comment t-16_b-16_k-32",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 16 
--comment t-16_b-16_k-64",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 16 
--comment t-16_b-16_k-256",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 16 
--comment t-16_b-16_k-64",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 128 --kl_beta 16 
--comment t-16_b-16_k-128",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 16 
--comment t-16_b-16_k-256",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 1024 --kl_beta 16 
--comment t-16_b-16_k-1024",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 16 --kl_beta 20.0 
--comment t-16_b-20_k-16",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 32 --kl_beta 20.0 
--comment t-16_b-20_k-32",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 64 --kl_beta 20.0 
--comment t-16_b-20_k-64",
        "./fit_model.sh '1' 'BALLS16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_latents 256 --kl_beta 20.0 
--comment t-16_b-20_k-256",
        "./fit_model.sh '1' 'BALLS64' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --n_lat